# Individuell uppgift – Hälsostudie, Del 1

I den här notebooken analyserar jag data från en hälsostudie med 800 deltagare.
Analysen omfattar beskrivande statistik, grafer, simulering, konfidensintervall och hypotesprövning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

## 1. Inläsning och första granskning av data

Kontroll av datasetets storlek, datatyper, saknade värden, dubbletter och vilka värden de kategoriska variablerna har.

**Observationer:**
- Datasetet innehåller 800 deltagare och 9 kolumner.
- Inga saknade värden förekommer i någon kolumn.
- Inga dubblerade ID:n och rader.
- Variablerna `sex` och `smoker` har endast två kategorier vardera, och
  `disease` är binär (0/1), vilket stämmer med uppgiftsbeskrivningen.
- Åldern sträcker sig mellan 18 och 90 år, medelvärdet för kolesterolnivå är ca 4,9 mmol/L, och medelvärdet för systoliskt blodtryck är ca 149 mmHg

**Anteckningar om kolesterolnivå och systoliskt blodtryck**
- Uppgiften anger endast "kolesterolnivå (mmol/L)". Eftersom medelvärdet (ca 4,9 mmol/L)
ligger nära vad som är typiskt för totalkolesterol, tolkar jag variabeln som
totalkolesterol. Det är en antagen tolkning och inte något som framgår av datan.
Som referens anges ett önskvärt värde för totalkolesterol ofta som under ca 5 mmol/L
(källa: https://www.1177.se/sjukdomar--besvar/hjarta-och-blodkarl/blodkarl/hoga-blodfetter/).

- Uppgiften anger blodtrycket som "systolic_bp" (mmHg). Medelvärdet i datasetet (ca 149 mmHg) ligger
högt jämfört med vad som anses normalt. Värden på 140 mmHg eller däröver räknas
vanligtvis som högt blodtryck (källa: https://www.1177.se/sjukdomar--besvar/hjarta-och-blodkarl/blodtryck/hogt-blodtryck/).

In [16]:
df = pd.read_csv("data/health_study_dataset.csv")

display(df.head())
df.info()
display(df.describe())

print("Antal dubbletter av id:", df["id"].duplicated().sum())
print("Antal dubbletter av rader (exklusive id):", df.drop(columns="id").duplicated().sum())

print("\nFördelning av kön:")
print(df["sex"].value_counts())
print("\nFördelning av rökare:")
print(df["smoker"].value_counts())
print("\nFördelning av sjukdom (0 = frisk, 1 = sjuk):")
print(df["disease"].value_counts())

,id,age,sex,height,weight,systolic_bp,cholesterol,smoker,disease
0,1,57,F,168.9,65.8,141.8,4.58,No,0
1,2,47,M,180.4,95.9,144.8,5.18,Yes,0
2,3,59,F,169.9,82.2,151.7,6.16,No,0
3,4,72,M,157.7,93.1,151.0,6.63,No,0
4,5,46,M,192.6,104.1,144.1,5.21,No,0


<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           800 non-null    int64  
 1   age          800 non-null    int64  
 2   sex          800 non-null    str    
 3   height       800 non-null    float64
 4   weight       800 non-null    float64
 5   systolic_bp  800 non-null    float64
 6   cholesterol  800 non-null    float64
 7   smoker       800 non-null    str    
 8   disease      800 non-null    int64  
dtypes: float64(4), int64(3), str(2)
memory usage: 56.4 KB


,id,age,height,weight,systolic_bp,cholesterol,disease
count,800.0000,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000
mean,400.5000,49.426250,171.849250,73.413000,149.178625,4.929150,0.058750
std,231.0844,14.501118,9.804259,13.685059,12.793360,0.848413,0.235303
min,1.0000,18.000000,144.400000,33.700000,106.800000,2.500000,0.000000
25%,200.7500,39.000000,164.775000,64.800000,140.900000,4.327500,0.000000
50%,400.5000,50.000000,171.350000,73.200000,149.400000,4.970000,0.000000
75%,600.2500,59.000000,178.925000,82.600000,157.600000,5.482500,0.000000
max,800.0000,90.000000,200.400000,114.400000,185.900000,7.880000,1.000000


Antal dubbletter av id: 0
Antal dubbletter av rader (exklusive id): 0

Fördelning av kön:
sex
F    403
M    397
Name: count, dtype: int64

Fördelning av rökare:
smoker
No     587
Yes    213
Name: count, dtype: int64

Fördelning av sjukdom (0 = frisk, 1 = sjuk):
disease
0    753
1     47
Name: count, dtype: int64


## 2. Beskrivande statistik

Medelvärde, median, minimum och maximum beräknas för de numeriska variablerna
age, weight, height, systolic_bp och cholesterol.

**Observationer:**
- Medelvärde och median ligger väldigt nära för värje variabel, som är ett karakteristik av  normalfördelning.
- Minsta värdet för vikt är 33.7 som är ganska ovanligt för vuxna, det bör undersökas vidare.
- Med referenser för normal kolesterol och systoliskt blodtryck som ligger ungefär < 5 mmol/L och < 140 mmHg respektive, medelvärden av ca 4.9 mmol/L och 149 mmHg visar att många undersökade ligger över normala gränsen.

In [17]:
num_col = ["age", "weight", "height", "systolic_bp", "cholesterol"]

stats = df[num_col].agg(["mean", "median", "min", "max"]).T
stats.columns = ["Medelvärde", "Median", "Min", "Max"]

display(stats.round(2))

,Medelvärde,Median,Min,Max
age,49.43,50.00,18.0,90.00
weight,73.41,73.20,33.7,114.40
height,171.85,171.35,144.4,200.40
systolic_bp,149.18,149.40,106.8,185.90
cholesterol,4.93,4.97,2.5,7.88


## Studiedesign och analysplan

- **Population:** Från beskrivande statistik kan man förstå att populationen omfattar vuxna personer i åldern 18–90 år.
- **Jämförelse mot kontrollgrupp:** Detta är inte ett experiment utan en
  observationsstudie. Jag jämför rökare med icke-rökare (kontrollgrupp). Ingen
  tilldelning har gjorts, så resultaten visar samband och inte orsakssamband.
- **Primär mätvariabel:** systolic_bp (mmHg).
- **Utfallets typ:** Kontinuerligt (systolic_bp). Variabeln disease är binär och
  analyseras separat i simuleringsdelen.
- **Praktiskt betydelsefull skillnad:** Jag valde 5 mmHg som praktiskt betydelsefull skillnad eftersom en sänkning på 5 mmHg anses vara kliniskt relevant enligt följande riktlinjer (källa: https://www.cochranelibrary.com/cdsr/doi/10.1002/14651858.CD015092.pub2/supplementarymaterials/CD015092-SUP-07-other.html).
- **Signifikansnivå:** α = 0,05.
- **Ensidigt eller tvåsidigt test:** Hypotesen "rökare har högre medelblodtryck" har
  en bestämd riktning, så jag använder ett ensidigt test.
- **Stickprovsstorlek:** 800 deltagare, varav 213 rökare och 587 icke-rökare.
- **Sekundära analyser:** Konfidensintervall för medelblodtrycket med två metoder,
  andel sjuka per rökstatus samt regressionsanalyser i del 2. Kolesterol används
  endast beskrivande och i grafer, och testas inte formellt. Dessa analyser
  betraktas som utforskande och inte som bekräftande.